# Chapter 7 Computational Lab
## Expectation: Construction and Fundamental Properties

This notebook accompanies Chapter 7 of *Probability Theory with Python and AI*.

The chapter constructs expectation from first principles. The primary object is **not** a density integral and not a Riemann--Stieltjes formula. The development is:

$$
\text{simple variables}
\longrightarrow
\text{supremum over simple minorants}
\longrightarrow
\text{convergence theorems}
\longrightarrow
\text{LOTUS and inequalities}
\longrightarrow
\text{tail / Stieltjes / density representations}.
$$

### Learning goals

By the end of the lab you should be able to:

1. compute expectation for a simple random variable;
2. interpret expectation as a probability-weighted mean;
3. construct canonical dyadic simple minorants;
4. see how simple variables approximate general non-negative variables;
5. work correctly with positive and negative parts;
6. distinguish finite expectation from an undefined $\infty-\infty$ expression;
7. verify monotone convergence, Fatou and dominated convergence in explicit models;
8. use linearity without assuming independence;
9. understand law invariance of expectation;
10. use LOTUS through the law $P_X$;
11. recover countable weighted sums as a special case;
12. apply Markov, Jensen, Cauchy--Schwarz, Hölder, Minkowski and Lyapunov inequalities;
13. compute expectation from survival functions and tail sums;
14. understand the later Riemann--Stieltjes and density formulas as representations of the already constructed expectation;
15. compute deductible, cap and truncation expectations;
16. audit AI-generated claims about expectation.

> **Methodological principle.** Indicators and simple variables are the finite building blocks; approximation and convergence theorems extend finite statements to general random variables.


## 0. Setup

The notebook uses exact rational arithmetic for discrete models and numerical integration only when the chapter itself moves to tail, Stieltjes or density representations.


In [ ]:
from fractions import Fraction
from itertools import combinations
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def fmt_fraction(x):
    x = Fraction(x)
    if x.denominator == 1:
        return str(x.numerator)
    return rf"\frac{{{x.numerator}}}{{{x.denominator}}}"


def simple_expectation(values, probs):
    values = list(values)
    probs = [Fraction(p) for p in probs]
    if len(values) != len(probs):
        raise ValueError("values and probabilities must have the same length.")
    if any(p < 0 for p in probs):
        raise ValueError("probabilities must be non-negative.")
    if sum(probs, Fraction(0, 1)) != 1:
        raise ValueError("probabilities must sum to one.")
    return sum((Fraction(v) * p for v, p in zip(values, probs)), Fraction(0, 1))


def positive_part(x):
    return max(x, 0)


def negative_part(x):
    return max(-x, 0)


def dyadic_minorant(x, n):
    return (math.floor((2**n) * min(x, n))) / (2**n)


def weighted_expectation(values, probs, transform=lambda x: x):
    return sum(
        transform(x) * p
        for x, p in zip(values, probs)
    )


def lp_norm(values, probs, p):
    return (
        sum(prob * abs(x) ** p for x, prob in zip(values, probs))
    ) ** (1 / p)


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))
    for line in latex_lines:
        display(Math(line))
    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Expectation tools are ready."
    "</div>"
))


## 1. Expectation of a simple random variable

If

$$
S=\sum_{i=1}^{m}s_i\mathbf 1_{A_i},
$$

where $A_1,\ldots,A_m$ form a measurable partition, define

$$
\boxed{
\mathbb E[S]
=
\sum_{i=1}^{m}s_iP(A_i).
}
$$

This is the starting point of the construction.


### Why this is called a mean

If $p_i=P(A_i)$, then

$$
p_i\ge0,
\qquad
\sum_i p_i=1,
$$

so

$$
\sum_i s_i p_i
$$

is exactly the ordinary weighted arithmetic mean of the possible values.

When all $m$ values are equally likely,

$$
p_i=\frac1m,
$$

and expectation reduces to

$$
\frac{s_1+\cdots+s_m}{m}.
$$


In [ ]:
mean_values = widgets.Text(value="1,2,3,4,5,6", description="values")
mean_probs = widgets.Text(value="1/6,1/6,1/6,1/6,1/6,1/6", description="probs")
mean_output = widgets.Output()


def parse_fraction_list(text):
    return [Fraction(x.strip()) for x in text.split(",") if x.strip()]


def parse_number_list(text):
    return [Fraction(x.strip()) for x in text.split(",") if x.strip()]


def update_weighted_mean(*_):
    with mean_output:
        clear_output(wait=True)

        try:
            values = parse_number_list(mean_values.value)
            probs = parse_fraction_list(mean_probs.value)
            EX = simple_expectation(values, probs)
        except Exception as exc:
            display(Markdown(f"**Input error:** {exc}"))
            return

        display(Math(r"\mathbb E[X]=" + fmt_fraction(EX)))

        if len(set(probs)) == 1:
            arithmetic = sum(values, Fraction(0, 1)) / len(values)
            display(Math(
                r"\text{arithmetic mean}=" + fmt_fraction(arithmetic)
            ))


for control in (mean_values, mean_probs):
    control.observe(update_weighted_mean, names="value")

display(widgets.VBox([
    mean_values,
    mean_probs,
    mean_output,
]))
update_weighted_mean()


### Fair die: the mean need not be a possible outcome

For a fair die,

$$
\mathbb E[X]
=
\frac{1+2+3+4+5+6}{6}
=
3.5.
$$

No single throw can equal $3.5$.

The later Strong Law of Large Numbers explains why this number is nevertheless a mean: for independent copies $X_1,X_2,\ldots$,

$$
\frac{X_1+\cdots+X_n}{n}
\xrightarrow{\mathrm{a.s.}}
\mathbb E[X].
$$

This is an interpretation of expectation, not its definition.


In [ ]:
die_n = widgets.IntSlider(
    value=1000, min=10, max=20000, step=10, description="throws"
)
die_seed = widgets.IntSlider(
    value=7, min=0, max=1000, description="seed"
)
die_output = widgets.Output()


def update_die_mean(*_):
    with die_output:
        clear_output(wait=True)

        rng = np.random.default_rng(die_seed.value)
        draws = rng.integers(1, 7, size=die_n.value)
        running = np.cumsum(draws) / np.arange(1, die_n.value + 1)

        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.plot(running)
        ax.axhline(3.5, linestyle="--")
        ax.set_xlabel("number of throws")
        ax.set_ylabel("running average")
        ax.set_title("Simulation only: empirical average near 3.5")
        plt.show()

        display(Markdown(
            "This simulation illustrates the later law-of-large-numbers interpretation. "
            "It is not used to define expectation."
        ))


for control in (die_n, die_seed):
    control.observe(update_die_mean, names="value")

display(widgets.VBox([
    widgets.HBox([die_n, die_seed]),
    die_output,
]))
update_die_mean()


## 2. The simple expectation is representation-independent

A simple random variable may have many indicator representations.

For example,

$$
S=2\mathbf 1_A+2\mathbf 1_B+5\mathbf 1_C
$$

can also be written as

$$
S=2\mathbf 1_{A\cup B}+5\mathbf 1_C.
$$

If

$$
P(A)=0.2,\qquad
P(B)=0.3,\qquad
P(C)=0.5,
$$

both representations give

$$
\mathbb E[S]=3.5.
$$

The common-refinement argument proves that this is always true.


In [ ]:
pA = Fraction(1, 5)
pB = Fraction(3, 10)
pC = Fraction(1, 2)

rep1 = 2*pA + 2*pB + 5*pC
rep2 = 2*(pA+pB) + 5*pC

display(Math(r"\mathbb E_1[S]=" + fmt_fraction(rep1)))
display(Math(r"\mathbb E_2[S]=" + fmt_fraction(rep2)))
display(Markdown(f"Same value: **{rep1 == rep2}**"))


### Basic simple-variable properties

For simple random variables $S,T$ and real $a,b$,

$$
\mathbb E[aS+bT]
=
a\mathbb E[S]+b\mathbb E[T].
$$

If $S\le T$,

$$
\mathbb E[S]\le\mathbb E[T].
$$

In particular,

$$
\mathbb E[c]=c,
\qquad
\mathbb E[\mathbf 1_A]=P(A).
$$


## 3. Historical problem: Huygens and fair value

Suppose a game pays $a$ or $b$, each with probability $1/2$.

Then

$$
\mathbb E[X]
=
\frac{a+b}{2}.
$$

The certain amount with the same expectation is therefore

$$
c=\frac{a+b}{2}.
$$

For $a=3$ and $b=7$,

$$
c=5.
$$


In [ ]:
huygens_a = widgets.FloatSlider(
    value=3, min=-10, max=20, step=1, description="a"
)
huygens_b = widgets.FloatSlider(
    value=7, min=-10, max=20, step=1, description="b"
)
huygens_output = widgets.Output()


def update_huygens(*_):
    with huygens_output:
        clear_output(wait=True)
        value = (huygens_a.value + huygens_b.value) / 2
        display(Math(
            r"\mathbb E[X]=\frac{a+b}{2}=" + f"{value:g}"
        ))


for control in (huygens_a, huygens_b):
    control.observe(update_huygens, names="value")

display(widgets.VBox([
    widgets.HBox([huygens_a, huygens_b]),
    huygens_output,
]))
update_huygens()


## 4. The supremum definition for non-negative variables

For $X\ge0$, define

$$
\mathcal S_X
=
\{
S:S\text{ is non-negative simple and }S\le X
\}.
$$

Then

$$
\boxed{
\mathbb E[X]
=
\sup_{S\in\mathcal S_X}\mathbb E[S].
}
$$

The value $+\infty$ is allowed.

This is the **primary definition** for general non-negative random variables in the chapter.


### Canonical dyadic approximation

For $X\ge0$, define

$$
X_n
=
2^{-n}
\left\lfloor
2^n(X\wedge n)
\right\rfloor.
$$

Then

$$
0\le X_n\le X_{n+1}\le X,
$$

and

$$
X_n\uparrow X.
$$

Each $X_n$ is a non-negative simple random variable.


In [ ]:
dyadic_x = widgets.FloatSlider(
    value=3.2, min=0.0, max=8.0, step=0.1, description="X"
)
dyadic_N = widgets.IntSlider(
    value=8, min=1, max=15, description="max n"
)
dyadic_output = widgets.Output()


def update_dyadic(*_):
    with dyadic_output:
        clear_output(wait=True)

        x = dyadic_x.value
        N = dyadic_N.value

        approximations = [dyadic_minorant(x, n) for n in range(1, N + 1)]

        display(Markdown(
            "| n | X_n | error X-X_n |\n"
            "|---:|---:|---:|\n"
            + "\n".join(
                f"| {n} | {xn:.6f} | {x-xn:.6f} |"
                for n, xn in enumerate(approximations, 1)
            )
        ))

        assert all(
            approximations[i] <= approximations[i+1] + 1e-12
            for i in range(len(approximations)-1)
        )
        assert all(xn <= x + 1e-12 for xn in approximations)


for control in (dyadic_x, dyadic_N):
    control.observe(update_dyadic, names="value")

display(widgets.VBox([
    widgets.HBox([dyadic_x, dyadic_N]),
    dyadic_output,
]))
update_dyadic()


### Methodological principle: indicators $\to$ simple variables $\to$ general variables

The dyadic approximation illustrates one of the standard proof architectures of modern probability:

$$
\boxed{
\mathbf 1_A
\longrightarrow
\text{simple random variables}
\longrightarrow
X_n\uparrow X
\longrightarrow
\text{limit theorem}
\longrightarrow
\text{general result}.
}
$$

A recurring four-step template is:

1. prove the statement for indicators or simple variables;
2. approximate a non-negative variable by increasing simple variables;
3. pass to the limit with Monotone Convergence, Fatou or Dominated Convergence;
4. extend to signed variables using $X=X^+-X^-$, truncation or domination.

This same architecture later reappears in LOTUS, independence factorization and conditional expectation.


### Representative dyadic minorants

For the three values $0.6$, $1.4$ and $3.2$, the first three canonical minorants are:

$$
\begin{array}{c|ccc}
X & X_1 & X_2 & X_3\\
\hline
0.6 & 0.5 & 0.5 & 0.5\\
1.4 & 1 & 1.25 & 1.375\\
3.2 & 1 & 2 & 3
\end{array}
$$


In [ ]:
representatives = [0.6, 1.4, 3.2]

for x in representatives:
    vals = [dyadic_minorant(x, n) for n in (1, 2, 3)]
    display(Markdown(
        f"$X={x}$ gives **{vals}**"
    ))


## 5. Signed variables and integrability

For a real-valued random variable,

$$
X^+=\max\{X,0\},
\qquad
X^-=\max\{-X,0\}.
$$

Then

$$
X=X^+-X^-,
\qquad
|X|=X^++X^-.
$$

Construct $\mathbb E[X^+]$ and $\mathbb E[X^-]$ by the non-negative definition.

If at least one is finite, define

$$
\boxed{
\mathbb E[X]
=
\mathbb E[X^+]-\mathbb E[X^-].
}
$$

If both are infinite, expectation is undefined.


### $L^1$

A random variable is integrable when

$$
\mathbb E|X|<\infty.
$$

Equivalently,

$$
\mathbb E[X^+]<\infty
\quad\text{and}\quad
\mathbb E[X^-]<\infty.
$$

The space of integrable random variables is

$$
L^1(\Omega,\mathcal F,P).
$$


In [ ]:
signed_values = widgets.Text(value="2,-1", description="values")
signed_probs = widgets.Text(value="3/4,1/4", description="probs")
signed_output = widgets.Output()


def update_signed(*_):
    with signed_output:
        clear_output(wait=True)

        try:
            values = [Fraction(x.strip()) for x in signed_values.value.split(",") if x.strip()]
            probs = parse_fraction_list(signed_probs.value)
        except Exception:
            display(Markdown("**Invalid input.**"))
            return

        if len(values) != len(probs) or sum(probs, Fraction(0, 1)) != 1:
            display(Markdown("**Use equal-length lists and probabilities summing to one.**"))
            return

        Ep = sum(Fraction(max(v, 0)) * p for v, p in zip(values, probs))
        Em = sum(Fraction(max(-v, 0)) * p for v, p in zip(values, probs))
        Eabs = Ep + Em
        EX = Ep - Em

        display(Math(r"\mathbb E[X^+]=" + fmt_fraction(Ep)))
        display(Math(r"\mathbb E[X^-]=" + fmt_fraction(Em)))
        display(Math(r"\mathbb E|X|=" + fmt_fraction(Eabs)))
        display(Math(r"\mathbb E[X]=" + fmt_fraction(EX)))


for control in (signed_values, signed_probs):
    control.observe(update_signed, names="value")

display(widgets.VBox([
    signed_values,
    signed_probs,
    signed_output,
]))
update_signed()


### Why symmetric cancellation is not the definition

An expression such as

$$
\lim_{n\to\infty}
\int_{-n}^{n}x\,dF_X(x)
$$

may converge because positive and negative divergences cancel.

That principal value is **not** an expectation if

$$
\mathbb E[X^+]
=
\mathbb E[X^-]
=
\infty.
$$

The positive and negative parts must be treated separately.


## 6. Monotone Convergence

If

$$
0\le X_n\uparrow X
\quad\text{almost surely},
$$

then

$$
\boxed{
\mathbb E[X_n]\uparrow\mathbb E[X].
}
$$

This is the central bridge from simple variables to arbitrary non-negative variables.


### Increasing events

Let

$$
P(\{k\})=2^{-k},
\qquad
k=1,2,\ldots,
$$

and

$$
A_n=\{1,\ldots,n\}.
$$

Then

$$
\mathbf 1_{A_n}\uparrow1,
$$

and

$$
\mathbb E[\mathbf 1_{A_n}]
=
P(A_n)
=
1-2^{-n}
\uparrow1.
$$


In [ ]:
mct_N = widgets.IntSlider(value=12, min=1, max=30, description="N")
mct_output = widgets.Output()


def update_mct(*_):
    with mct_output:
        clear_output(wait=True)

        N = mct_N.value
        ns = np.arange(1, N + 1)
        expectations = 1 - 2.0 ** (-ns)

        fig, ax = plt.subplots(figsize=(8, 3.4))
        ax.plot(ns, expectations, marker="o")
        ax.axhline(1, linestyle="--")
        ax.set_xlabel("n")
        ax.set_ylabel("E[1_A_n]")
        ax.set_ylim(0, 1.03)
        ax.set_title("Monotone convergence for increasing events")
        plt.show()


mct_N.observe(update_mct, names="value")
display(widgets.VBox([mct_N, mct_output]))
update_mct()


## 7. Linearity does not require independence

For non-negative $X,Y$ and $a,b\ge0$,

$$
\mathbb E[aX+bY]
=
a\mathbb E[X]+b\mathbb E[Y].
$$

For $X,Y\in L^1$ and arbitrary real $a,b$, the same formula holds.

Independence is **not** a hypothesis.


In [ ]:
p = Fraction(2, 5)

# X = 1_A and Y = 1-X are maximally dependent.
EX = p
EY = 1 - p
EXYsum = Fraction(1, 1)

display(Math(r"\mathbb E[X]=" + fmt_fraction(EX)))
display(Math(r"\mathbb E[Y]=" + fmt_fraction(EY)))
display(Math(r"\mathbb E[X+Y]=1"))
display(Math(
    r"\mathbb E[X]+\mathbb E[Y]=" + fmt_fraction(EX + EY)
))


### Order and absolute bounds

For integrable $X,Y$:

$$
X\le Y\ \text{a.s.}
\Longrightarrow
\mathbb E[X]\le\mathbb E[Y],
$$

$$
|\mathbb E[X]|
\le
\mathbb E|X|,
$$

and if

$$
m\le X\le M\ \text{a.s.},
$$

then

$$
m\le\mathbb E[X]\le M.
$$


In [ ]:
values = [-2, 1]
probs = [Fraction(1, 4), Fraction(3, 4)]

EX = sum(Fraction(x) * p for x, p in zip(values, probs))
Eabs = sum(Fraction(abs(x)) * p for x, p in zip(values, probs))

display(Math(r"\mathbb E[X]=" + fmt_fraction(EX)))
display(Math(r"\mathbb E|X|=" + fmt_fraction(Eabs)))
display(Markdown(f"Absolute bound verified: **{abs(EX) <= Eabs}**"))


### Almost-sure and law invariance

If

$$
X=Y\quad\text{a.s.},
$$

then their expectations agree whenever defined.

More generally, if $X$ and $Y$ have the same law, then their positive and negative expectations agree separately. Thus expectation depends only on the law.


In [ ]:
# Same Bernoulli(1/3) law on two different finite spaces.
law1_values = [1, 0, 0]
law1_probs = [Fraction(1, 3)] * 3

law2_values = [1, 1, 0, 0, 0, 0]
law2_probs = [Fraction(1, 6)] * 6

E1 = sum(Fraction(x) * p for x, p in zip(law1_values, law1_probs))
E2 = sum(Fraction(x) * p for x, p in zip(law2_values, law2_probs))

display(Math(r"\mathbb E[X]=" + fmt_fraction(E1)))
display(Math(r"\mathbb E[Y]=" + fmt_fraction(E2)))
display(Markdown(f"Law-invariant expectation: **{E1 == E2}**"))


## 8. Fatou's lemma

For non-negative random variables,

$$
\boxed{
\mathbb E\left[
\liminf_{n\to\infty}X_n
\right]
\le
\liminf_{n\to\infty}\mathbb E[X_n].
}
$$

Fatou is a one-sided limit theorem. Equality need not hold.


### Strict Fatou inequality

On $\Omega=\mathbb N$ with

$$
P(\{k\})=2^{-k},
$$

let

$$
X_n=2^n\mathbf 1_{\{n\}}.
$$

For every fixed outcome,

$$
X_n\to0,
$$

but

$$
\mathbb E[X_n]=1
$$

for every $n$. Hence

$$
0
=
\mathbb E[\liminf X_n]
<
\liminf\mathbb E[X_n]
=
1.
$$


In [ ]:
fatou_N = widgets.IntSlider(value=12, min=1, max=25, description="N")
fatou_output = widgets.Output()


def update_fatou(*_):
    with fatou_output:
        clear_output(wait=True)

        N = fatou_N.value
        expectations = np.ones(N)

        fig, ax = plt.subplots(figsize=(8, 3.0))
        ax.plot(range(1, N + 1), expectations, marker="o")
        ax.axhline(0, linestyle="--")
        ax.set_ylim(-0.05, 1.15)
        ax.set_xlabel("n")
        ax.set_ylabel("E[X_n]")
        ax.set_title("E[X_n]=1 while X_n converges pointwise to 0")
        plt.show()


fatou_N.observe(update_fatou, names="value")
display(widgets.VBox([fatou_N, fatou_output]))
update_fatou()


## 9. Dominated Convergence

Suppose

$$
X_n\to X
\quad\text{almost surely},
$$

and there is $Y\in L^1$ such that

$$
|X_n|\le Y
\quad\text{a.s. for every }n.
$$

Then

$$
\boxed{
\mathbb E|X_n-X|\to0,
}
$$

and consequently

$$
\boxed{
\mathbb E[X_n]\to\mathbb E[X].
}
$$


### Signed dominated sequence

On $\Omega=\mathbb N$ with $P(\{k\})=2^{-k}$, define

$$
X_n
=
(-1)^n
\mathbf 1_{\{n,n+1,\ldots\}}.
$$

Then

$$
X_n\to0,
\qquad
|X_n|\le1,
$$

and

$$
\mathbb E[X_n]
=
(-1)^n2^{1-n}
\to0.
$$


In [ ]:
dct_N = widgets.IntSlider(value=14, min=2, max=30, description="N")
dct_output = widgets.Output()


def update_dct(*_):
    with dct_output:
        clear_output(wait=True)

        N = dct_N.value
        ns = np.arange(1, N + 1)
        expectations = ((-1.0) ** ns) * (2.0 ** (1 - ns))

        fig, ax = plt.subplots(figsize=(8, 3.4))
        ax.plot(ns, expectations, marker="o")
        ax.axhline(0, linestyle="--")
        ax.set_xlabel("n")
        ax.set_ylabel("E[X_n]")
        ax.set_title("Dominated convergence in a signed example")
        plt.show()


dct_N.observe(update_dct, names="value")
display(widgets.VBox([dct_N, dct_output]))
update_dct()


## 10. Expectation of a transformation: LOTUS

Write

$$
\mu_X(B)=P(X\in B)
$$

for the law of $X$.

For non-negative Borel $g$, the law integral is built by the same simple-minorant construction:

$$
\int_{\mathbb R}g\,d\mu_X
=
\sup_{0\le s\le g}
\int_{\mathbb R}s\,d\mu_X.
$$

Then

$$
\boxed{
\mathbb E[g(X)]
=
\int_{\mathbb R}g(x)\,d\mu_X(x).
}
$$

For signed $g$, the same identity holds whenever the positive and negative parts define the expression.


### One expectation, several computational forms

There are not separate concepts of “discrete expectation” and “continuous expectation.”

There is one expectation. Depending on the law, the law integral may reduce to:

$$
\sum_k g(x_k)p_k
$$

for a countable law, or to

$$
\int_{\mathbb R}g(x)f_X(x)\,dx
$$

when a density exists.

The general law integral remains meaningful for mixed laws and laws that are neither purely discrete nor absolutely continuous.


In [ ]:
lotus_values = [-1, 2]
lotus_probs = [Fraction(1, 3), Fraction(2, 3)]

Eg = sum(
    Fraction(x*x) * p
    for x, p in zip(lotus_values, lotus_probs)
)

Eh = sum(
    Fraction(int(x <= 0)) * p
    for x, p in zip(lotus_values, lotus_probs)
)

display(Math(r"\mathbb E[X^2]=" + fmt_fraction(Eg)))
display(Math(r"\mathbb E[\mathbf 1_{\{X\le0\}}]=" + fmt_fraction(Eh)))


### Law integral and Stieltjes notation

The cdf $F_X$ generates the law $\mu_X$. Therefore the same general law integral can be written in Lebesgue--Stieltjes notation as

$$
\boxed{
\int_{\mathbb R}g\,d\mu_X
=
\int_{\mathbb R}g\,dF_X.
}
$$

For continuous $g$ on compact intervals, the classical Riemann--Stieltjes integral agrees with the Lebesgue--Stieltjes integral. The latter is the more general notion.


## 11. Countably valued random variables

If $X$ takes values $x_1,x_2,\ldots$ with probabilities $p_k$, then

$$
\mathbb E[X^+]
=
\sum_k x_k^+p_k,
$$

$$
\mathbb E[X^-]
=
\sum_k x_k^-p_k.
$$

Thus

$$
X\in L^1
\iff
\sum_k|x_k|p_k<\infty,
$$

and then

$$
\boxed{
\mathbb E[X]
=
\sum_kx_kp_k.
}
$$


### Example

If

$$
P(X=k)=2^{-k},
\qquad
k=1,2,\ldots,
$$

then

$$
\mathbb E[X]
=
\sum_{k=1}^{\infty}\frac{k}{2^k}
=
2.
$$


In [ ]:
series_N = widgets.IntSlider(value=10, min=1, max=40, description="N")
series_output = widgets.Output()


def update_countable_series(*_):
    with series_output:
        clear_output(wait=True)

        N = series_N.value
        partial = sum(k / (2**k) for k in range(1, N + 1))

        display(Math(
            r"\sum_{k=1}^{" + str(N) + r"}\frac{k}{2^k}"
            + f"\approx {partial:.8f}"
        ))
        display(Math(r"\mathbb E[X]=2"))


series_N.observe(update_countable_series, names="value")
display(widgets.VBox([series_N, series_output]))
update_countable_series()


## 12. Markov's inequality

If $X\ge0$ and $a>0$, then

$$
\boxed{
P(X\ge a)
\le
\frac{\mathbb E[X]}{a}.
}
$$

The proof is the pointwise inequality

$$
X
\ge
a\mathbf 1_{\{X\ge a\}}.
$$


In [ ]:
markov_mean = widgets.FloatSlider(
    value=2400, min=0, max=10000, step=100, description="E[X]"
)
markov_a = widgets.FloatSlider(
    value=12000, min=100, max=30000, step=100, description="a"
)
markov_output = widgets.Output()


def update_markov(*_):
    with markov_output:
        clear_output(wait=True)

        if markov_a.value <= 0:
            return

        bound = min(1.0, markov_mean.value / markov_a.value)
        display(Math(
            r"P(X\ge a)\le\frac{\mathbb E[X]}{a}\le"
            + f"{bound:.4f}"
        ))


for control in (markov_mean, markov_a):
    control.observe(update_markov, names="value")

display(widgets.VBox([
    widgets.HBox([markov_mean, markov_a]),
    markov_output,
]))
update_markov()


### Sharpness of Markov's inequality

If $0\le m\le a$, define

$$
P(X=a)=\frac{m}{a},
\qquad
P(X=0)=1-\frac{m}{a}.
$$

Then

$$
\mathbb E[X]=m
$$

and

$$
P(X\ge a)
=
\frac{m}{a}
=
\frac{\mathbb E[X]}{a}.
$$

So Markov's inequality can be exact.


## 13. Jensen's inequality

Let $\varphi$ be convex, and suppose $X$ and $\varphi(X)$ are integrable. Then

$$
\boxed{
\varphi(\mathbb E[X])
\le
\mathbb E[\varphi(X)].
}
$$

For $\varphi(x)=x^2$,

$$
(\mathbb E[X])^2
\le
\mathbb E[X^2].
$$


In [ ]:
j_values = [-1, 2]
j_probs = [0.5, 0.5]

EX = sum(x*p for x, p in zip(j_values, j_probs))
EX2 = sum((x**2)*p for x, p in zip(j_values, j_probs))

display(Math(r"(\mathbb E[X])^2=" + f"{EX**2:.4f}"))
display(Math(r"\mathbb E[X^2]=" + f"{EX2:.4f}"))
display(Markdown(f"Jensen verified: **{EX**2 <= EX2}**"))


## 14. $L^p$ spaces and moment size

For $p>0$,

$$
L^p
=
\{X:\mathbb E|X|^p<\infty\},
$$

and

$$
\|X\|_p
=
\bigl(\mathbb E|X|^p\bigr)^{1/p}.
$$

For $p\ge1$, after identifying almost-surely equal variables, this is a norm.

For $0<p<1$, the same expression is useful for moment comparison but is not a norm in general.


In [ ]:
lp_p = widgets.FloatSlider(
    value=2.0, min=0.5, max=5.0, step=0.5, description="p"
)
lp_output = widgets.Output()


def update_lp(*_):
    with lp_output:
        clear_output(wait=True)

        values = [0.0, 2.0]
        probs = [0.5, 0.5]
        p = lp_p.value
        norm = lp_norm(values, probs, p)

        display(Math(
            r"\|X\|_p=" + f"{norm:.6f}"
        ))

        if p < 1:
            display(Markdown("For this range the expression is not a norm in general."))


lp_p.observe(update_lp, names="value")
display(widgets.VBox([lp_p, lp_output]))
update_lp()


### Cauchy--Schwarz

If $U,V\in L^2$, then

$$
\boxed{
|\mathbb E[UV]|^2
\le
\mathbb E[U^2]\mathbb E[V^2].
}
$$

Equality holds exactly when $U$ and $V$ are linearly dependent almost surely.


In [ ]:
# Finite three-state example
probs = np.array([0.2, 0.3, 0.5])
U = np.array([1.0, -1.0, 2.0])
V = np.array([2.0, 0.0, 1.0])

EUV = np.sum(probs * U * V)
EU2 = np.sum(probs * U**2)
EV2 = np.sum(probs * V**2)

display(Math(r"|\mathbb E[UV]|^2=" + f"{EUV**2:.6f}"))
display(Math(
    r"\mathbb E[U^2]\mathbb E[V^2]=" + f"{EU2*EV2:.6f}"
))
display(Markdown(f"Cauchy--Schwarz verified: **{EUV**2 <= EU2*EV2 + 1e-12}**"))


### Hölder

For conjugate exponents $p,q>1$ satisfying

$$
\frac1p+\frac1q=1,
$$

$$
\boxed{
\mathbb E|XY|
\le
\|X\|_p\|Y\|_q.
}
$$

Cauchy--Schwarz is the special case $p=q=2$.


In [ ]:
holder_p = widgets.FloatSlider(
    value=3.0, min=1.1, max=5.0, step=0.1, description="p"
)
holder_output = widgets.Output()


def update_holder(*_):
    with holder_output:
        clear_output(wait=True)

        p = holder_p.value
        q = p / (p - 1)

        probs = np.array([0.2, 0.3, 0.5])
        X = np.array([1.0, -2.0, 3.0])
        Y = np.array([2.0, 1.0, -1.0])

        lhs = np.sum(probs * np.abs(X * Y))
        norm_x = np.sum(probs * np.abs(X)**p) ** (1/p)
        norm_y = np.sum(probs * np.abs(Y)**q) ** (1/q)
        rhs = norm_x * norm_y

        display(Math(r"q=" + f"{q:.6f}"))
        display(Math(r"\mathbb E|XY|=" + f"{lhs:.6f}"))
        display(Math(r"\|X\|_p\|Y\|_q=" + f"{rhs:.6f}"))
        display(Markdown(f"Hölder verified: **{lhs <= rhs + 1e-12}**"))


holder_p.observe(update_holder, names="value")
display(widgets.VBox([holder_p, holder_output]))
update_holder()


### Minkowski

For $p\ge1$,

$$
\boxed{
\|X+Y\|_p
\le
\|X\|_p+\|Y\|_p.
}
$$

This is the triangle inequality that turns $L^p$ into a normed space modulo almost-sure equality.


In [ ]:
minkowski_p = widgets.FloatSlider(
    value=2.0, min=1.0, max=5.0, step=0.5, description="p"
)
minkowski_output = widgets.Output()


def update_minkowski(*_):
    with minkowski_output:
        clear_output(wait=True)

        p = minkowski_p.value
        probs = np.array([0.2, 0.3, 0.5])
        X = np.array([1.0, -2.0, 3.0])
        Y = np.array([2.0, 1.0, -1.0])

        norm_sum = np.sum(probs * np.abs(X + Y)**p) ** (1/p)
        norm_x = np.sum(probs * np.abs(X)**p) ** (1/p)
        norm_y = np.sum(probs * np.abs(Y)**p) ** (1/p)

        display(Math(r"\|X+Y\|_p=" + f"{norm_sum:.6f}"))
        display(Math(r"\|X\|_p+\|Y\|_p=" + f"{norm_x+norm_y:.6f}"))
        display(Markdown(f"Minkowski verified: **{norm_sum <= norm_x+norm_y + 1e-12}**"))


minkowski_p.observe(update_minkowski, names="value")
display(widgets.VBox([minkowski_p, minkowski_output]))
update_minkowski()


### Lyapunov moment inequality

If

$$
0<r<s
$$

and $\mathbb E|X|^s<\infty$, then

$$
\boxed{
\|X\|_r
\le
\|X\|_s.
}
$$

On a probability space,

$$
L^s\subseteq L^r
\qquad
(1\le r<s).
$$


In [ ]:
lyap_r = widgets.FloatSlider(value=1.0, min=0.5, max=3.0, step=0.5, description="r")
lyap_s = widgets.FloatSlider(value=2.0, min=1.0, max=5.0, step=0.5, description="s")
lyap_output = widgets.Output()


def update_lyapunov(*_):
    with lyap_output:
        clear_output(wait=True)

        r = lyap_r.value
        s = lyap_s.value

        if not r < s:
            display(Markdown("**Choose r<s.**"))
            return

        values = [0.0, 2.0]
        probs = [0.5, 0.5]

        nr = lp_norm(values, probs, r)
        ns = lp_norm(values, probs, s)

        display(Math(r"\|X\|_r=" + f"{nr:.6f}"))
        display(Math(r"\|X\|_s=" + f"{ns:.6f}"))
        display(Markdown(f"Lyapunov verified: **{nr <= ns + 1e-12}**"))


for control in (lyap_r, lyap_s):
    control.observe(update_lyapunov, names="value")

display(widgets.VBox([
    widgets.HBox([lyap_r, lyap_s]),
    lyap_output,
]))
update_lyapunov()


## 15. Expectation as an area under a tail

For $X\ge0$,

$$
\boxed{
\mathbb E[X]
=
\int_0^\infty P(X>x)\,dx
=
\int_0^\infty(1-F_X(x))\,dx.
}
$$

This is proved from the simple-variable formula and monotone approximation before the later Stieltjes representation.


### Survival-area example

Suppose

$$
F_X(x)=
\begin{cases}
0,&x<0,\\
x/2,&0\le x<2,\\
1,&x\ge2.
\end{cases}
$$

Then

$$
P(X>x)=1-\frac{x}{2}
$$

for $0\le x<2$, and

$$
\mathbb E[X]
=
\int_0^2
\left(1-\frac{x}{2}\right)dx
=
1.
$$


In [ ]:
tail_grid = np.linspace(0, 2, 500)
survival_values = 1 - tail_grid / 2

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(tail_grid, survival_values)
ax.fill_between(tail_grid, 0, survival_values, alpha=0.2)
ax.set_xlabel("x")
ax.set_ylabel("P(X>x)")
ax.set_title("Expectation as survival area")
plt.show()

area = np.trapz(survival_values, tail_grid)
display(Math(r"\text{numerical area}\approx" + f"{area:.6f}"))


### Two-sided tail formula

For a real-valued $X$,

$$
\mathbb E[X^+]
=
\int_0^\infty P(X>x)\,dx,
$$

$$
\mathbb E[X^-]
=
\int_0^\infty P(X<-x)\,dx.
$$

Thus, when $X$ is integrable,

$$
\boxed{
\mathbb E[X]
=
\int_0^\infty P(X>x)\,dx
-
\int_0^\infty P(X<-x)\,dx.
}
$$

Also,

$$
X\in L^1
\iff
\int_0^\infty P(|X|>x)\,dx<\infty.
$$


In [ ]:
# X=-1 with prob 1/4, X=2 with prob 3/4
positive_area = 2 * Fraction(3, 4)
negative_area = 1 * Fraction(1, 4)

display(Math(r"\text{positive tail area}=" + fmt_fraction(positive_area)))
display(Math(r"\text{negative tail area}=" + fmt_fraction(negative_area)))
display(Math(r"\mathbb E[X]=" + fmt_fraction(positive_area-negative_area)))


### Tail-sum formula for integer-valued variables

If $X$ takes values in $\mathbb N_0$, then

$$
\boxed{
\mathbb E[X]
=
\sum_{k=0}^{\infty}P(X>k)
=
\sum_{k=1}^{\infty}P(X\ge k).
}
$$


In [ ]:
tail_sum_N = widgets.IntSlider(value=12, min=1, max=40, description="N")
tail_sum_output = widgets.Output()


def update_tail_sum(*_):
    with tail_sum_output:
        clear_output(wait=True)

        N = tail_sum_N.value

        # P(X=k)=2^{-(k+1)}, k>=0, so P(X>k)=2^{-(k+1)}.
        partial = sum(2 ** (-(k+1)) for k in range(N + 1))

        display(Math(
            r"\sum_{k=0}^{" + str(N) + r"}P(X>k)"
            + f"={partial:.8f}"
        ))
        display(Math(r"\mathbb E[X]=1"))


tail_sum_N.observe(update_tail_sum, names="value")
display(widgets.VBox([tail_sum_N, tail_sum_output]))
update_tail_sum()


## 16. Riemann--Stieltjes representation comes later

After the expectation operator, convergence theorems, LOTUS, inequalities and tail formulas have already been built from the supremum construction, define the Riemann--Stieltjes candidate

$$
\mathbb E_{\!RS}[X^+]
=
\int_{-\infty}^{\infty}x^+\,dF_X(x),
$$

$$
\mathbb E_{\!RS}[X^-]
=
\int_{-\infty}^{\infty}x^-\,dF_X(x).
$$

The chapter proves that this representation agrees with the primary supremum-based expectation.


### Three forms, one value

Suppose

$$
P(X=0)=\frac14,
\qquad
P(X=2)=\frac34.
$$

The simple formula gives

$$
\mathbb E[X]
=
\frac32.
$$

The cdf has a jump $3/4$ at $2$, so the Stieltjes form contributes

$$
2\cdot\frac34
=
\frac32.
$$

The survival function is $3/4$ on $[0,2)$, so

$$
\int_0^\infty P(X>x)\,dx
=
\int_0^2\frac34\,dx
=
\frac32.
$$


In [ ]:
simple_value = 2 * Fraction(3, 4)
stieltjes_value = 2 * Fraction(3, 4)
tail_value = 2 * Fraction(3, 4)

display(Math(r"\text{simple form}=" + fmt_fraction(simple_value)))
display(Math(r"\text{Stieltjes form}=" + fmt_fraction(stieltjes_value)))
display(Math(r"\text{tail form}=" + fmt_fraction(tail_value)))
display(Markdown(f"All equal: **{simple_value == stieltjes_value == tail_value}**"))


### Equivalent representations for a non-negative variable

Once equivalence has been proved,

$$
\boxed{
\mathbb E[X]
=
\sup_{\substack{S\text{ non-negative simple}\\S\le X}}
\mathbb E[S]
=
\int_{-\infty}^{\infty}x^+\,dF_X(x)
=
\int_0^\infty(1-F_X(x))\,dx.
}
$$

The first expression is the construction; the other two are representations.


## 17. Density representation

If the law has a density $f_X$, then

$$
\mathbb E[X^+]
=
\int_{-\infty}^{\infty}x^+f_X(x)\,dx,
$$

$$
\mathbb E[X^-]
=
\int_{-\infty}^{\infty}x^-f_X(x)\,dx.
$$

Hence

$$
X\in L^1
\iff
\int_{-\infty}^{\infty}|x|f_X(x)\,dx<\infty,
$$

and then

$$
\boxed{
\mathbb E[X]
=
\int_{-\infty}^{\infty}x f_X(x)\,dx.
}
$$

This is a computational representation of the general expectation, not a separate definition.


### Example

For

$$
f_X(x)=
\begin{cases}
2x,&0\le x\le1,\\
0,&\text{otherwise},
\end{cases}
$$

$$
\mathbb E[X]
=
\int_0^1x(2x)\,dx
=
\frac23,
$$

and

$$
\mathbb E[X^2]
=
\int_0^1x^2(2x)\,dx
=
\frac12.
$$

Jensen for $\varphi(x)=x^2$ gives

$$
\frac49
\le
\frac12.
$$


In [ ]:
density_grid = np.linspace(0, 1, 10001)
density = 2 * density_grid

EX = np.trapz(density_grid * density, density_grid)
EX2 = np.trapz((density_grid**2) * density, density_grid)

display(Math(r"\mathbb E[X]\approx" + f"{EX:.8f}"))
display(Math(r"\mathbb E[X^2]\approx" + f"{EX2:.8f}"))
display(Markdown(f"Jensen numerically verified: **{EX**2 <= EX2 + 1e-10}**"))


### Density version of LOTUS

If $X$ has density $f_X$ and

$$
\int_{\mathbb R}|g(x)|f_X(x)\,dx<\infty,
$$

then

$$
\boxed{
\mathbb E[g(X)]
=
\int_{\mathbb R}g(x)f_X(x)\,dx.
}
$$

Again, this is a special representation of the general law-integral formula.


## 18. Mixed laws: continuous mass and atoms in one expectation

Suppose a cdf has:

- density slope $0.4$ on $(0,1)$;
- an atom of mass $0.3$ at $1$;
- density slope $0.1$ on $(1,3)$;
- an atom of mass $0.1$ at $3$.

Then

$$
\begin{aligned}
\mathbb E[X]
&=
0.4\int_0^1x\,dx
+
0.1\int_1^3x\,dx\\
&\quad
+
0.3(1)
+
0.1(3)\\
&=
1.2.
\end{aligned}
$$

This example illustrates why the general law integral is useful: continuous and atomic contributions are handled in one framework.


In [ ]:
mixed_cont_1 = 0.4 * 0.5
mixed_cont_2 = 0.1 * ((3**2 - 1**2) / 2)
mixed_atoms = 0.3*1 + 0.1*3
mixed_mean = mixed_cont_1 + mixed_cont_2 + mixed_atoms

display(Math(r"\mathbb E[X]=" + f"{mixed_mean:.6f}"))
assert abs(mixed_mean - 1.2) < 1e-12


## 19. Threshold and truncation expectations

For $X\ge0$ and $d,u>0$,

$$
\mathbb E[(X-d)^+]
=
\int_d^\infty P(X>x)\,dx,
$$

$$
\mathbb E[X\wedge u]
=
\int_0^uP(X>x)\,dx,
$$

and for

$$
T_d^u=\min\{(X-d)^+,u\},
$$

$$
\boxed{
\mathbb E[T_d^u]
=
\int_d^{d+u}P(X>x)\,dx.
}
$$


### Decomposition

Pointwise,

$$
X
=
X\wedge d
+
(X-d)^+.
$$

Therefore, whenever the expectations are defined,

$$
\boxed{
\mathbb E[X]
=
\mathbb E[X\wedge d]
+
\mathbb E[(X-d)^+].
}
$$


In [ ]:
# Mixed-cdf survival from the chapter.
def mixed_survival(x):
    if x < 0:
        return 1.0
    if x < 1:
        return 1 - 0.4*x
    if x < 3:
        return 0.4 - 0.1*x
    return 0.0


grid1 = np.linspace(0, 1, 5001)
grid2 = np.linspace(1, 3, 5001)

EXwedge1 = np.trapz([mixed_survival(x) for x in grid1], grid1)
excess1 = np.trapz([mixed_survival(x) for x in grid2], grid2)

grid_cap = np.linspace(1, 2, 3001)
cap11 = np.trapz([mixed_survival(x) for x in grid_cap], grid_cap)

display(Math(r"\mathbb E[X\wedge1]\approx" + f"{EXwedge1:.6f}"))
display(Math(r"\mathbb E[(X-1)^+]\approx" + f"{excess1:.6f}"))
display(Math(r"\mathbb E[T_1^1]\approx" + f"{cap11:.6f}"))
display(Math(r"\mathbb E[X]=1.2"))


### Survival-area geometry of a deductible and cap

For the capped positive part,

$$
T_d^u
=
\min\{(X-d)^+,u\},
$$

the expectation is the area under the survival curve between $d$ and $d+u$.


In [ ]:
area_d = widgets.FloatSlider(value=2.0, min=0.0, max=6.0, step=0.5, description="d")
area_u = widgets.FloatSlider(value=4.0, min=0.5, max=6.0, step=0.5, description="u")
area_output = widgets.Output()


def update_survival_area(*_):
    with area_output:
        clear_output(wait=True)

        d = area_d.value
        u = area_u.value

        grid = np.linspace(0, 12, 1000)
        surv = np.exp(-grid/3)

        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.plot(grid, surv)
        mask = (grid >= d) & (grid <= d + u)
        ax.fill_between(grid[mask], 0, surv[mask], alpha=0.2)
        ax.axvline(d, linestyle="--")
        ax.axvline(d+u, linestyle=":")
        ax.set_xlabel("threshold x")
        ax.set_ylabel("survival")
        ax.set_title("Capped positive-part expectation as survival area")
        plt.show()

        exact = 3*(math.exp(-d/3) - math.exp(-(d+u)/3))
        display(Math(r"\mathbb E[T_d^u]=" + f"{exact:.6f}"))


for control in (area_d, area_u):
    control.observe(update_survival_area, names="value")

display(widgets.VBox([
    widgets.HBox([area_d, area_u]),
    area_output,
]))
update_survival_area()


## 20. Distribution specified by its survival function

Let $X\ge0$ have

$$
\overline F_X(x)=
\begin{cases}
1,&0\le x<1,\\
x^{-2},&x\ge1.
\end{cases}
$$

Then

$$
\mathbb E[X]
=
\int_0^1 1\,dx
+
\int_1^\infty x^{-2}\,dx
=
2.
$$

Also,

$$
\mathbb E[(X-2)^+]
=
\int_2^\infty x^{-2}\,dx
=
\frac12,
$$

and

$$
\mathbb E[\min\{(X-2)^+,3\}]
=
\int_2^5x^{-2}\,dx
=
\frac3{10}.
$$


In [ ]:
display(Math(r"\mathbb E[X]=2"))
display(Math(r"\mathbb E[(X-2)^+]=\frac12"))
display(Math(r"\mathbb E[\min\{(X-2)^+,3\}]=\frac3{10}"))


## 21. Computational verification: mixed law

The next experiment approximates a mixed distribution with continuous pieces and atoms using cdf increments.

The key numerical rule is:

> do **not** differentiate a cdf near jumps; use increments $F(x_i)-F(x_{i-1})$ so continuous mass and atoms enter through the same Stieltjes sum.


In [ ]:
def mixed_cdf(x):
    if x < 0:
        return 0.0
    if x < 1:
        return 0.4*x
    if x < 3:
        return 0.7 + 0.1*(x-1)
    return 1.0


def stieltjes_sum_mean(n_intervals):
    grid = np.linspace(0, 3, n_intervals + 1)

    # Include atom locations explicitly.
    grid = np.unique(np.concatenate([grid, np.array([1.0, 3.0])]))
    grid.sort()

    Fvals = np.array([mixed_cdf(x) for x in grid])
    increments = np.diff(Fvals)
    right_tags = grid[1:]

    return float(np.sum(right_tags * increments))


def tail_mean(n_intervals):
    grid = np.linspace(0, 3, n_intervals + 1)
    survival = 1 - np.array([mixed_cdf(x) for x in grid])
    return float(np.trapz(survival, grid))


verification_N = widgets.IntSlider(
    value=800, min=50, max=5000, step=50, description="intervals"
)
verification_output = widgets.Output()


def update_verification(*_):
    with verification_output:
        clear_output(wait=True)

        N = verification_N.value
        rs = stieltjes_sum_mean(N)
        tail = tail_mean(N)

        display(Math(r"\text{Stieltjes approximation}\approx" + f"{rs:.6f}"))
        display(Math(r"\text{tail-area approximation}\approx" + f"{tail:.6f}"))
        display(Math(r"\text{exact mean}=1.2"))


verification_N.observe(update_verification, names="value")
display(widgets.VBox([verification_N, verification_output]))
update_verification()


## 22. Guided exercise generator

The exercises follow the chapter's order: construction first, representation second.


In [ ]:
exercise_rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random", "random"),
        ("Simple expectation", "simple"),
        ("Positive/negative parts", "signed"),
        ("Linearity", "linearity"),
        ("Markov", "markov"),
        ("Jensen", "jensen"),
        ("Tail formula", "tail"),
        ("LOTUS", "lotus"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_output = widgets.Output()
feedback_output = widgets.Output()
exercise_state = {}


def make_exercise(_=None):
    kind = exercise_kind.value

    if kind == "random":
        kind = exercise_rng.choice([
            "simple", "signed", "linearity", "markov", "jensen", "tail", "lotus"
        ])

    if kind == "simple":
        target = "5"
        prompt = "A fair game pays 3 or 7. Find its expectation."
        hint = "Take the probability-weighted mean."
        solution = r"\mathbb E[X]=(3+7)/2=5."

    elif kind == "signed":
        target = "1.25"
        prompt = "P(X=2)=3/4 and P(X=-1)=1/4. Find E[X]."
        hint = "Compute positive and negative contributions separately."
        solution = r"\mathbb E[X]=3/2-1/4=5/4."

    elif kind == "linearity":
        target = "no"
        prompt = "Does linearity of expectation require independence? yes/no"
        hint = "Linearity is structural and does not involve a product."
        solution = r"\text{No.}"

    elif kind == "markov":
        target = "0.2"
        prompt = "If X>=0 and E[X]=2400, give Markov's bound for P(X>=12000)."
        hint = "Divide E[X] by the threshold."
        solution = r"P(X\ge12000)\le0.2."

    elif kind == "jensen":
        target = "yes"
        prompt = "For convex phi, does Jensen say phi(E[X]) <= E[phi(X)]? yes/no"
        hint = "Recall the supporting-line argument."
        solution = r"\text{Yes.}"

    elif kind == "tail":
        target = "1"
        prompt = "If P(X>x)=1-x/2 on 0<=x<2 and 0 afterwards, find E[X]."
        hint = "Integrate the survival function."
        solution = r"\int_0^2(1-x/2)\,dx=1."

    else:
        target = "3"
        prompt = "P(X=-1)=1/3 and P(X=2)=2/3. Find E[X^2]."
        hint = "Apply LOTUS to g(x)=x^2."
        solution = r"\mathbb E[X^2]=1/3+8/3=3."

    exercise_state.clear()
    exercise_state.update(
        target=target,
        hint=hint,
        solution=solution,
    )
    answer_box.value = ""

    with prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))
    with feedback_output:
        clear_output(wait=True)


def show_hint(_):
    with feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + exercise_state["hint"]))


def reveal_solution(_):
    with feedback_output:
        clear_output(wait=True)
        display(Math(exercise_state["solution"]))


def check_answer(_):
    with feedback_output:
        clear_output(wait=True)

        guess = answer_box.value.strip().lower().replace(" ", "")
        target = exercise_state["target"].replace(" ", "")

        try:
            if target not in {"yes", "no"} and abs(float(guess) - float(target)) < 1e-8:
                display(Markdown("**Correct.**"))
                return
        except Exception:
            pass

        display(Markdown(
            "**Correct.**"
            if guess == target
            else "**Not yet. Identify the relevant theorem before computing.**"
        ))


new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal_solution)
check_button.on_click(check_answer)

display(widgets.VBox([
    widgets.HBox([exercise_kind, new_button]),
    prompt_output,
    widgets.HBox([answer_box, check_button]),
    widgets.HBox([hint_button, reveal_button]),
    feedback_output,
]))

make_exercise()


## 23. AI Audit: expectation claims

Audit any AI-generated argument using the following checklist:

1. Is expectation being defined from the correct starting point?
2. For $X\ge0$, is $+\infty$ allowed?
3. For signed $X$, are $X^+$ and $X^-$ treated separately?
4. Is an $\infty-\infty$ expression incorrectly assigned a value?
5. Is linearity being incorrectly made dependent on independence?
6. Is a pointwise or almost-sure convergence theorem being used without its hypotheses?
7. In Monotone Convergence, is the sequence genuinely increasing and non-negative?
8. In Dominated Convergence, is one integrable dominating variable present?
9. Is Fatou being used as an equality rather than a one-sided inequality?
10. Is LOTUS being restricted unnecessarily to density models?
11. Is the law integral being confused with only the classical Riemann--Stieltjes integral?
12. Is a countable weighted sum being presented as a separate definition rather than a special case?
13. Is Jensen being replaced by the false identity $\mathbb E[g(X)]=g(\mathbb E[X])$?
14. Are the $L^p$ hypotheses of Hölder or Minkowski satisfied?
15. Is a principal value being confused with expectation?
16. Are density and Stieltjes formulas presented as representations rather than foundations?
17. Are deductible/cap formulas matched to the correct survival interval?

### Three claims to audit

- “Linearity of expectation requires independence.”
- “If a symmetric principal value integral equals zero, the expectation exists and equals zero.”
- “For any nonlinear $g$, $\mathbb E[g(X)]=g(\mathbb E[X])$.”

All three are false.


### Suggested AI audit prompts

- “Ask me to prove expectation for indicators, then simple variables, then extend by monotone approximation.”
- “Give a sequence for which Fatou's inequality is strict and make me verify every expectation.”
- “Construct a dominated sequence and require explicit verification of the dominating $L^1$ variable.”
- “Give the same law on two different sample spaces and test law invariance of expectation.”
- “Create a mixed law with both a density part and atoms, then compute its mean from cdf increments.”
- “Give a survival function and ask for $\mathbb E[X]$, $\mathbb E[(X-d)^+]$, $\mathbb E[X\wedge u]$ and $\mathbb E[T_d^u]$.”
- “Make me explain why density formulas and Riemann--Stieltjes formulas come after the supremum construction.”


## 24. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. Primary definition for X>=0:",
        [
            "Choose...",
            "density integral",
            "supremum over non-negative simple minorants",
            "symmetric truncation",
        ],
        "supremum over non-negative simple minorants",
        r"\mathbb E[X]=\sup_{0\le S\le X}\mathbb E[S].",
    ),
    (
        "2. For X>=0, E[X] may equal +infinity:",
        ["Choose...", "true", "false"],
        "true",
        r"\text{The non-negative expectation takes values in }[0,\infty].",
    ),
    (
        "3. Linearity requires independence:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{Linearity is valid without independence.}",
    ),
    (
        "4. If E[X+]=E[X-]=infinity, then E[X]:",
        ["Choose...", "is 0", "is undefined", "is infinity"],
        "is undefined",
        r"\infty-\infty\text{ is not assigned a value.}",
    ),
    (
        "5. Monotone Convergence requires:",
        [
            "Choose...",
            "non-negative increasing sequence",
            "independence",
            "identical distributions",
        ],
        "non-negative increasing sequence",
        r"0\le X_n\uparrow X\Longrightarrow\mathbb E[X_n]\uparrow\mathbb E[X].",
    ),
    (
        "6. Fatou's lemma is generally:",
        ["Choose...", "an equality", "a one-sided inequality"],
        "a one-sided inequality",
        r"\mathbb E[\liminf X_n]\le\liminf\mathbb E[X_n].",
    ),
    (
        "7. LOTUS requires a density:",
        ["Choose...", "true", "false"],
        "false",
        r"\mathbb E[g(X)]=\int g\,d\mu_X\text{ is formulated for the law itself.}",
    ),
    (
        "8. Jensen for convex phi says:",
        [
            "Choose...",
            "phi(E[X])<=E[phi(X)]",
            "phi(E[X])=E[phi(X)] always",
            "phi(E[X])>=E[phi(X)]",
        ],
        "phi(E[X])<=E[phi(X)]",
        r"\varphi(\mathbb E[X])\le\mathbb E[\varphi(X)].",
    ),
    (
        "9. For X>=0, the tail formula is:",
        [
            "Choose...",
            "E[X]=integral_0^infinity P(X>x) dx",
            "E[X]=P(X>0)",
        ],
        "E[X]=integral_0^infinity P(X>x) dx",
        r"\mathbb E[X]=\int_0^\infty P(X>x)\,dx.",
    ),
    (
        "10. The density formula in this chapter is:",
        [
            "Choose...",
            "the fundamental definition",
            "a representation after the general construction",
        ],
        "a representation after the general construction",
        r"\text{The construction precedes density and Stieltjes formulas.}",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="440px"),
    )
    quiz_widgets.append(dropdown)
    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:660px'>{prompt}</div>"),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)

        score = sum(
            widget.value == correct
            for widget, (_, _, correct, _) in zip(
                quiz_widgets,
                quiz_data,
            )
        )

        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))

        for i, (
            widget,
            (_, _, correct, explanation),
        ) in enumerate(zip(quiz_widgets, quiz_data), 1):
            mark = "✓" if widget.value == correct else "✗"
            display(Markdown(
                f"**{mark} Question {i}:** correct answer = `{correct}`"
            ))
            display(Math(explanation))


grade_button.on_click(grade_quiz)
display(widgets.VBox(
    quiz_rows + [grade_button, quiz_output]
))


## 25. Automatic mathematical verification

The final code cell checks representative identities from the construction, inequalities and computational representations.


In [ ]:
# Simple expectation and Huygens.
assert simple_expectation(
    [3, 7],
    [Fraction(1, 2), Fraction(1, 2)],
) == 5

# Signed parts.
values = [2, -1]
probs = [Fraction(3, 4), Fraction(1, 4)]

Ep = sum(Fraction(max(x, 0)) * p for x, p in zip(values, probs))
Em = sum(Fraction(max(-x, 0)) * p for x, p in zip(values, probs))
assert Ep == Fraction(3, 2)
assert Em == Fraction(1, 4)
assert Ep - Em == Fraction(5, 4)

# Dyadic minorants.
for x in [0.0, 0.6, 1.4, 3.2, 6.7]:
    seq = [dyadic_minorant(x, n) for n in range(1, 12)]
    assert all(a <= b + 1e-12 for a, b in zip(seq, seq[1:]))
    assert all(v <= x + 1e-12 for v in seq)

# Linearity in an exactly dependent pair X=1_A, Y=1-X.
p = Fraction(2, 5)
assert p + (1-p) == 1

# Jensen quadratic example.
vals = np.array([-1.0, 2.0])
pr = np.array([0.5, 0.5])
EX = np.sum(pr * vals)
EX2 = np.sum(pr * vals**2)
assert EX**2 <= EX2 + 1e-12

# Cauchy-Schwarz finite example.
pr = np.array([0.2, 0.3, 0.5])
U = np.array([1.0, -1.0, 2.0])
V = np.array([2.0, 0.0, 1.0])
EUV = np.sum(pr * U * V)
EU2 = np.sum(pr * U**2)
EV2 = np.sum(pr * V**2)
assert EUV**2 <= EU2*EV2 + 1e-12

# Hölder, p=3 and q=3/2.
X = np.array([1.0, -2.0, 3.0])
Y = np.array([2.0, 1.0, -1.0])
p = 3.0
q = 1.5

lhs = np.sum(pr * np.abs(X*Y))
rhs = (
    np.sum(pr * np.abs(X)**p) ** (1/p)
    * np.sum(pr * np.abs(Y)**q) ** (1/q)
)
assert lhs <= rhs + 1e-12

# Minkowski, p=2.
p = 2.0
norm_sum = np.sum(pr * np.abs(X+Y)**p) ** (1/p)
norm_x = np.sum(pr * np.abs(X)**p) ** (1/p)
norm_y = np.sum(pr * np.abs(Y)**p) ** (1/p)
assert norm_sum <= norm_x + norm_y + 1e-12

# Lyapunov, r=1, s=2.
vals = [0.0, 2.0]
pr2 = [0.5, 0.5]
assert lp_norm(vals, pr2, 1) <= lp_norm(vals, pr2, 2)

# Countable law partial sums approach 2.
for N in [5, 10, 20]:
    partial = sum(k / (2**k) for k in range(1, N+1))
    assert partial < 2 + 1e-12

# Tail forms.
assert 2 * Fraction(3, 4) - Fraction(1, 4) == Fraction(5, 4)

# Mixed law expectation.
mixed_mean = (
    0.4 * 0.5
    + 0.1 * ((3**2 - 1**2)/2)
    + 0.3*1
    + 0.1*3
)
assert abs(mixed_mean - 1.2) < 1e-12

# Density example.
assert abs((2/3)**2 - 4/9) < 1e-12
assert 4/9 <= 1/2

# Threshold decomposition for mixed example.
assert abs(0.8 + 0.4 - 1.2) < 1e-12
assert abs(0.25 - 0.25) < 1e-12

show_result(
    "All Chapter 7 automatic checks passed",
    r"\mathbb E[S]=\sum_i s_iP(A_i)",
    r"0\le X_n\uparrow X",
    r"\mathbb E[aX+bY]=a\mathbb E[X]+b\mathbb E[Y]",
    r"\mathbb E[g(X)]=\int g\,d\mu_X",
    r"\varphi(\mathbb E[X])\le\mathbb E[\varphi(X)]",
    r"\mathbb E[X]=\int_0^\infty P(X>x)\,dx\quad(X\ge0)",
    note="Construction-first expectation, inequalities and later representations are numerically consistent."
)


## 26. Chapter map

| Chapter concept | Computational representation |
|---|---|
| simple expectation | exact weighted mean |
| “expectation as mean” | fair-die weighted mean and simulation illustration |
| well-definedness | equivalent simple representations |
| Huygens fair value | two-outcome weighted mean |
| non-negative expectation | supremum viewpoint |
| dyadic approximation | interactive $X_n$ table |
| methodological pipeline | indicator $\to$ simple $\to$ approximation $\to$ limit |
| signed expectation | positive and negative parts |
| $L^1$ | finite absolute expectation |
| Monotone Convergence | increasing-event example |
| linearity | dependent complementary indicators |
| law invariance | same law on different sample spaces |
| Fatou | strict inequality example |
| Dominated Convergence | signed shrinking tails |
| LOTUS | law integral, no density required |
| countable law | weighted series |
| Markov | tail-bound widget |
| Jensen | quadratic example |
| $L^p$ | moment-size widget |
| Cauchy--Schwarz | exact finite model |
| Hölder | conjugate-power widget |
| Minkowski | triangle inequality widget |
| Lyapunov | monotonicity of moment size |
| tail integral | survival-area plot |
| two-sided tail formula | positive minus negative tail areas |
| integer tail sum | cumulative-tail series |
| Riemann--Stieltjes representation | equality with constructed expectation |
| density formula | continuous computational form |
| mixed law | density pieces plus atoms |
| deductible/cap expectations | survival area over $[d,d+u]$ |
| AI Audit | hypothesis and definition checking |

The conceptual hierarchy is:

$$
\boxed{
\text{construction first}
\;\longrightarrow\;
\text{theory}
\;\longrightarrow\;
\text{representations}
\;\longrightarrow\;
\text{computation}.
}
$$
